# 01 - Landing & Bronze: Transacties (ING CSV)

Dit notebook doet de eerste stap van de medallion pipeline voor het `transacties` domein:

1. Schema's aanmaken in DuckDB (`meta`, `landing`, `bronze`, `silver`, `gold`)
2. Bestandslog bijhouden (`meta.landing_bestanden_log`) o.b.v. content-hash, om dubbele import te voorkomen
3. ING CSV bestanden inlezen als `dtype=str` (geen type-casting in bronze)
4. Per-rij SHA-256 hash toevoegen (traceerbaarheid / dedup op rijniveau)
5. Wegschrijven naar `bronze.transacties_ing` binnen een transactie (fail-fast)

**Verwacht ING CSV formaat** (pas de kolomnamen in dit notebook aan als jouw export anders is):
`Datum, Naam / Omschrijving, Rekening, Tegenrekening, Code, Af Bij, Bedrag (EUR), Mutatiesoort, Mededelingen`


In [ ]:
import sys
from pathlib import Path


def find_project_root(marker="requirements.txt") -> Path:
    p = Path.cwd().resolve()
    for kandidaat in [p, *p.parents]:
        if (kandidaat / marker).exists():
            return kandidaat
    raise RuntimeError("Kon project root niet vinden")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import duckdb
from src.pipeline import schema
from src.pipeline.paths import DB_PAD
from src.pipeline.transacties import bronze

con = duckdb.connect(str(DB_PAD))
print(f"Verbonden met {DB_PAD}")

## 1. Schema's en bronze-tabel initialiseren

Schema's, `meta.landing_bestanden_log` en `bronze.transacties_ing` worden aangemaakt door `schema.init_schemas` en `bronze.run_bronze` — zie `src/pipeline/schema.py` en `src/pipeline/transacties/bronze.py`.

In [ ]:
schema.init_schemas(con)
print("Schema's + meta.landing_bestanden_log klaar")

## 2. Alle nieuwe bestanden in de landingmap verwerken

In [ ]:
resultaat = bronze.run_bronze(con)
print(resultaat)

## 3. Verificatie

In [ ]:
print("Bestandslog:")
display(con.execute("SELECT * FROM meta.landing_bestanden_log ORDER BY verwerkt_op DESC").df())

print("\nAantal rijen in bronze.transacties_ing:")
print(con.execute("SELECT COUNT(*) AS aantal FROM bronze.transacties_ing").df())

print("\nSteekproef:")
display(con.execute("SELECT * FROM bronze.transacties_ing LIMIT 5").df())


In [ ]:
con.close()